# Fig. 6C: pseudotime transferred onto human nuclei, on the joint SAMap UMAP

Each human nucleus colored by the mouse pseudotime transferred to it through the SAMap alignment (the transfer of
record, `transferred_mm_dpt_pseudotime_ranknorm`: the edge-weighted mean over its valued mouse neighbors), on the
joint SAMap UMAP, on black. Mouse nuclei sit underneath in dark grey for context; a second version leaves them out.
Human nuclei without a valued mouse neighbor are drawn larger in grey behind the colored ones; they are not imputed.
The square frame is fitted to the whole joint embedding, so every UMAP panel of Fig. 6 shares it.

Display rules, as in the methods and the SAMap supplement: one fixed 0-1 viridis scale with values as stored,
never re-ranked. Every nucleus is drawn.

**Inputs**: `joint.parquet` (rule `samap_figure_tables`: species, joint UMAP, 10x library and transferred values of every
nucleus) and the human fibroblast/decidual plot table of the release whose pseudotime was transferred (QC'd
Slide-tags positions, phase).
**Outputs**, in `OUT`: `umap_dark_transferred_pseudotime.{png,svg,pdf}` and
`umap_dark_transferred_pseudotime_humanonly.{png,svg,pdf}` (90 x 90 mm, 600 dpi) and
`fig6c_umap_transferred_pseudotime.provenance.json` with the counts drawn.

Fonts: Helvetica Neue where installed (macOS, the version of record), otherwise Helvetica, otherwise DejaVu Sans; the
provenance record names the font used.

**Running it.** Run all cells with this directory as the working directory: the shared modules next to the notebook
are imported from there. The cells above *Run* only define constants and functions. Paths are relative to the
repository root. Existing outputs are not overwritten unless `FORCE = True`.

The pipeline runs it with papermill (`workflow/rules/10c_samap_figures.smk`). By hand, from the repository root:

```bash
papermill notebooks/samap/fig6c_umap_transferred_pseudotime.ipynb \
    results/notebooks/samap/fig6c_umap_transferred_pseudotime.executed.ipynb \
    --cwd notebooks/samap -p FORCE True \
    -p HUMAN_TABLE results/plot_tables/<release>/human_fibdec_plot_table.parquet
```

Commit this notebook without outputs: the provenance record notes whether the notebook matches its commit.

In [ ]:
# Parameters. papermill -p NAME VALUE overrides them in a cell below this one.
# Paths are relative to the repository root; REPO = None finds the root with git from the working directory.
REPO = None
JOINT = "results/samap_figures/tables/joint.parquet"
HUMAN_TABLE = None  # results/plot_tables/<release>/human_fibdec_plot_table.parquet; the rule passes it
OUT = "results/figures/samap_main"
FORCE = False  # overwrite existing outputs
NOTEBOOK = "notebooks/samap/fig6c_umap_transferred_pseudotime.ipynb"  # this notebook, for the provenance record

In [ ]:
import datetime
import hashlib
import json
import logging
import platform
import subprocess
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from matplotlib import font_manager
from matplotlib.lines import Line2D

# This notebook's messages in the cell output; other libraries stay at WARNING.
logging.basicConfig(format="%(message)s", stream=sys.stdout, force=True)
LOG = logging.getLogger("fig6c")
LOG.setLevel(logging.INFO)

# The human plot table has no default: it is the release of record's (rule samap_fibdec_plot_table), which
# the rule passes; a run by hand passes it too (-p HUMAN_TABLE ...).
if HUMAN_TABLE is None:
    raise ValueError("set HUMAN_TABLE: -p HUMAN_TABLE results/plot_tables/<release>/human_fibdec_plot_table.parquet")

## Constants

In [ ]:
TRANSFERRED = "transferred_mm_dpt_pseudotime_ranknorm"  # the transfer of record, in joint.parquet

# Fixed 0-1 viridis with values as stored: a transfer that flattens the values shows as flat color.
CMAP = mpl.colormaps["viridis"]
NORM = mpl.colors.Normalize(0, 1)

# Helvetica Neue is a system font on macOS that matplotlib does not find on its own; elsewhere the family list
# falls through to DejaVu Sans.
FONT_FAMILY = ["Helvetica Neue", "Helvetica", "DejaVu Sans"]
MACOS_HELVETICA_NEUE = Path("/System/Library/Fonts/HelveticaNeue.ttc")
TEXT = "white"

VARIANTS = {
    "with_mouse": "umap_dark_transferred_pseudotime",
    "human_only": "umap_dark_transferred_pseudotime_humanonly",
}
TITLE = "Pseudotime transferred onto human cells"

MOUSE_CONTEXT = "#3a3a3a"  # mouse nuclei: their shape shows, their color does not compete
NO_VALUE = "#9a9a9a"       # human nuclei without a valued mouse SAMap neighbor
AXIS_ARROWS = "#bbbbbb"

PANEL_MM = 90
UMAP_DOT = 0.5             # pt^2
NO_VALUE_DOT_SCALE = 3     # larger, so the few nuclei without a value stay visible
DPI = 600
ORDER_SEED = 3             # seeded random draw order, so neither early nor late nuclei always sit on top

## Figure

In [ ]:
def set_font() -> str:
    """Register Helvetica Neue on macOS and set the family list; returns the family that will be used."""
    if MACOS_HELVETICA_NEUE.exists():
        font_manager.fontManager.addfont(str(MACOS_HELVETICA_NEUE))
    mpl.rcParams.update({"font.family": FONT_FAMILY, "svg.fonttype": "none", "pdf.fonttype": 42})
    return font_manager.FontProperties(family=FONT_FAMILY).get_name()


def human_nuclei(joint: pd.DataFrame, human_table_path: Path) -> pd.DataFrame:
    """Human nuclei of the joint export with their position and phase from the release."""
    human_table = pd.read_parquet(human_table_path, columns=["x_um_qc", "y_um_qc", "timepoint"])
    # timepoint comes from the release table; the joint export carries its own copy.
    human = joint[joint.species == "hs"].drop(columns=["timepoint"]).join(human_table, how="inner")
    if len(human) != (joint.species == "hs").sum():
        raise ValueError("human nuclei in the joint export are missing from the release table")
    return human


def find_repo(repo) -> Path:
    if repo is not None:
        return Path(repo)
    git_root = ["git", "rev-parse", "--show-toplevel"]
    return Path(subprocess.run(git_root, capture_output=True, text=True, check=True).stdout.strip())


def file_record(path: Path, role: str) -> dict:
    stat = path.stat()
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return {
        "path": str(path.resolve()),
        "role": role,
        "bytes": stat.st_size,
        "mtime": datetime.datetime.fromtimestamp(stat.st_mtime).isoformat(timespec="seconds"),
        "sha256": digest.hexdigest(),
    }


def git_revision(path: Path):
    """Commit of the checkout holding path, and whether path matches it; None outside git."""
    git = ["git", "-C", str(path.parent)]
    try:
        commit = subprocess.run(git + ["rev-parse", "HEAD"], capture_output=True, text=True, check=True)
        status = subprocess.run(git + ["status", "--porcelain", "--", path.name],
                                capture_output=True, text=True, check=True)
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None
    return {"commit": commit.stdout.strip(), "notebook_matches_commit": status.stdout.strip() == ""}


def square_limits(umap: pd.DataFrame) -> tuple:
    """Square frame around the full joint embedding (both species), 6% wider than its larger extent."""
    x, y = umap["X_umap__0"], umap["X_umap__1"]
    side = max(x.max() - x.min(), y.max() - y.min()) * 1.06
    cx, cy = (x.max() + x.min()) / 2, (y.max() + y.min()) / 2
    return (cx - side / 2, cx + side / 2), (cy - side / 2, cy + side / 2), side


def umap_figure(title: str, xlim, ylim, side):
    """Black square panel with a boxless equal-aspect UMAP axes and short axis arrows."""
    fig = plt.figure(figsize=(PANEL_MM / 25.4, PANEL_MM / 25.4), facecolor="black")
    ax = fig.add_axes([0.06, 0.06, 0.88, 0.84])
    ax.set_facecolor("black")
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    fig.text(0.06, 0.965, title, ha="left", va="top", fontsize=9, color=TEXT)

    # UMAP coordinates have no units, so two short arrows replace a box and ticks.
    length = 0.14 * side
    arrow = dict(arrowprops=dict(arrowstyle="-|>", lw=0.8, color=AXIS_ARROWS, mutation_scale=6, shrinkA=0,
                                 shrinkB=0),
                 annotation_clip=False)
    ax.annotate("", xy=(xlim[0] + length, ylim[0]), xytext=(xlim[0], ylim[0]), **arrow)
    ax.annotate("", xy=(xlim[0], ylim[0] + length), xytext=(xlim[0], ylim[0]), **arrow)
    ax.text(xlim[0] + 0.015 * side, ylim[0] - 0.012 * side, "UMAP 1", ha="left", va="top", fontsize=7,
            color=AXIS_ARROWS)
    ax.text(xlim[0] - 0.015 * side, ylim[0] + 0.015 * side, "UMAP 2", ha="right", va="bottom", rotation=90,
            fontsize=7, color=AXIS_ARROWS)
    return fig, ax


def scatter_umap(ax, cells: pd.DataFrame, colors, zorder: int, size: float = UMAP_DOT) -> None:
    ax.scatter(cells["X_umap__0"], cells["X_umap__1"], s=size, c=colors, linewidths=0, rasterized=True,
               zorder=zorder)


def colorbar_vertical(fig) -> None:
    cax = fig.add_axes([0.80, 0.66, 0.025, 0.20])
    bar = fig.colorbar(mpl.cm.ScalarMappable(norm=NORM, cmap=CMAP), cax=cax, ticks=[0, 0.5, 1])
    bar.ax.set_yticklabels(["0 early", "0.5", "1 late"], fontsize=7, color=TEXT)
    bar.outline.set_visible(False)
    bar.ax.tick_params(length=1.5, pad=1.5, color=TEXT)
    cax.set_title("pseudotime\n(rank)", fontsize=7, loc="left", pad=4, color=TEXT)


def legend_bottom_right(fig, entries) -> None:
    """Key for the grey nuclei, in the empty bottom-right corner of the embedding."""
    handles = [Line2D([], [], ls="", marker="o", ms=4, mfc=color, mec="none", label=label)
               for color, label in entries]
    kax = fig.add_axes([0.56, 0.0, 0.44, 0.16])
    kax.axis("off")
    kax.legend(handles=handles, loc="lower left", fontsize=7, handletextpad=0.3, labelspacing=0.4, borderaxespad=0,
               labelcolor=TEXT, frameon=False)

## Run

In [ ]:
repo = find_repo(REPO)
notebook = repo / NOTEBOOK
if not notebook.is_file():
    raise FileNotFoundError(f"{notebook} not found; set NOTEBOOK to this notebook's path")

out = repo / OUT
files = {variant: [out / f"{stem}.{ext}" for ext in ("png", "svg", "pdf")] for variant, stem in VARIANTS.items()}
sidecar = out / "fig6c_umap_transferred_pseudotime.provenance.json"
outputs = [path for paths in files.values() for path in paths]
existing = [str(path) for path in outputs + [sidecar] if path.exists()]
if existing and not FORCE:
    raise FileExistsError(f"Refusing to overwrite {existing}; set FORCE = True")
out.mkdir(parents=True, exist_ok=True)

In [ ]:
joint = pd.read_parquet(repo / JOINT)
human = human_nuclei(joint, repo / HUMAN_TABLE)
mouse = joint[joint.species == "mm"]

has_value = human[TRANSFERRED].notna()
counts = {
    "umap_human_with_value": int(has_value.sum()),
    "umap_human_no_value": int((~has_value).sum()),
    "umap_mouse_context": len(mouse),
}
LOG.info("Nuclei: %s", counts)

In [ ]:
font = set_font()
LOG.info("Font: %s", font)

# Every frame shares one square fitted to the whole joint embedding.
xlim, ylim, side = square_limits(joint)
colored = human[has_value].sample(frac=1, random_state=ORDER_SEED)
# Grey nuclei have no mouse neighbor, or only decAp neighbors, which carry no pseudotime; the key names what they
# lack rather than why.
no_value_label = f"no transferred\nvalue ({counts['umap_human_no_value']})"
legends = {
    "with_mouse": [(MOUSE_CONTEXT, "mouse cells"), (NO_VALUE, no_value_label)],
    "human_only": [(NO_VALUE, no_value_label)],
}

for variant in VARIANTS:
    fig, ax = umap_figure(TITLE, xlim, ylim, side)
    if variant == "with_mouse":
        scatter_umap(ax, mouse, MOUSE_CONTEXT, zorder=1)

    # Human nuclei without a value sit behind the colored ones, above the mouse context.
    scatter_umap(ax, human[~has_value], NO_VALUE, zorder=2, size=UMAP_DOT * NO_VALUE_DOT_SCALE)
    scatter_umap(ax, colored, CMAP(NORM(colored[TRANSFERRED].to_numpy())), zorder=3)
    colorbar_vertical(fig)
    legend_bottom_right(fig, legends[variant])
    for path in files[variant]:
        fig.savefig(path, dpi=DPI, facecolor="black")
    plt.close(fig)
    LOG.info("Wrote %s", [str(path) for path in files[variant]])

display(Image(filename=str(files["with_mouse"][0]), width=420))

In [ ]:
parameters = {"REPO": str(repo), "JOINT": JOINT, "HUMAN_TABLE": HUMAN_TABLE, "OUT": OUT, "FORCE": FORCE}

record = {
    "outputs": [str(path.resolve()) for path in outputs],
    "written": datetime.datetime.now().isoformat(timespec="seconds"),
    "host": platform.node(),
    "notebook": str(notebook.resolve()),
    "notebook_md5": hashlib.md5(notebook.read_bytes()).hexdigest(),
    "git": git_revision(notebook),
    "parameters": parameters,
    "inputs": [file_record(repo / JOINT, "joint"), file_record(repo / HUMAN_TABLE, "human_table")],
    "font": font,
    "versions": {"python": platform.python_version(), "matplotlib": mpl.__version__, "pandas": pd.__version__},
    "transferred_column": TRANSFERRED,
    "rescaled_or_reranked": False,
    "colormap": "viridis, fixed 0-1",
}

record["umap"] = counts | {"order_seed": ORDER_SEED, "panel_mm": PANEL_MM, "dpi": DPI}
sidecar.write_text(json.dumps(record, indent=1, default=str))
LOG.info("Wrote %s", sidecar)